In [15]:
%pip install openpyxl
%pip install --upgrade pip


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ICA is OpenAI-compatible. The SDK appends "chat/completions" to base_url,
# and the trailing slash matters — without it, httpx's URL joiner replaces
# the last segment and you get 404 Not Found.
base_url = os.getenv("ANTHROPIC_BASE_URL", "").rstrip("/") + "/chat-models/"

client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=base_url,
)

model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
response = client.chat.completions.create(
    model=model,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": "Customer wants to return an order. Write a short helpful response",
        },
        {
            "role": "assistant",
            "content": "What is your order number",
        },
        {
            "role": "user",
            "content": "My order number is 12345",
        }
    ],
)
print(response.choices[0].message.content)
print(f"\nfinish_reason: {response.choices[0].finish_reason}")

Thank you for providing your order number! Here is a short helpful response for the customer:

---

**Dear Customer,**

Thank you for reaching out to us! We are sorry to hear that you would like to return your order.

We have located your order **#12345** in our system. To process your return, please follow these steps:

1. **Confirm the reason** for your return
2. **Ensure the item** is in its original condition and packaging
3. **Ship the item** back to us using the return label we will email to you
4. **Allow 5-7 business days** for your refund to be processed once we receive the item

If you have any questions or need further assistance, please don't hesitate to contact us.

**We apologize for any inconvenience and hope to serve you better in the future!**

Best Regards,
**Customer Support Team**

---

Is there anything else you would like me to add or change? 😊

finish_reason: stop


In [9]:
#Response method for multiple requests

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ICA is OpenAI-compatible. The SDK appends "chat/completions" to base_url,
# and the trailing slash matters — without it, httpx's URL joiner replaces
# the last segment and you get 404 Not Found.
base_url = os.getenv("ANTHROPIC_BASE_URL", "").rstrip("/") + "/chat-models/"

client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=base_url,
)
model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")
def Add_User_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def Add_Assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def Get_Chat_Response(messages):
    response = client.chat.completions.create(
        model=model,
        max_tokens=300,
        messages=messages,
    )
    return response.choices[0].message.content, response.choices[0].finish_reason

In [ ]:
#Execute the multi-request chat
messages = []

Add_User_message(messages, "I want to return an order")
chat_response = Get_Chat_Response(messages)
Add_Assistant_message(messages, chat_response[0])
Add_User_message(messages, "My order number is 12345")
chat_response = Get_Chat_Response(messages)
Add_Assistant_message(messages, chat_response[0])
print(messages)

[{'role': 'user', 'content': 'I want to return an order'}, {'role': 'assistant', 'content': 'I\'d be happy to help you with your return! However, I should let you know that **I\'m an AI assistant and don\'t have access to any order management systems** or your account information.\n\nTo process your return, here are some steps you can take:\n\n## Common Ways to Initiate a Return:\n\n1. **Online Portal** – Log into your account on the retailer\'s website and look for "Orders" or "Returns" section.\n\n2. **Contact Customer Service** – Reach out directly to the retailer via:\n   - 📞 Phone\n   - 💬 Live chat\n   - 📧 Email\n\n3. **Visit a Store** – If it\'s a brick-and-mortar retailer, you may be able to return in person.\n\n---\n\nTo help you further, could you tell me:\n- **Which retailer/company** did you order from?\n- **What is the issue** with the order? (wrong item, damaged, changed mind, etc.)\n\nWith that information, I can guide you on the **specific return policy** or steps for th

In [ ]:
#Apply the excel update methods
import os
from dotenv import load_dotenv
from openai import OpenAI
from __future__ import annotations
from datetime import datetime
from typing import Any, Optional, Tuple
from openpyxl import Workbook, load_workbook

load_dotenv()

# ICA is OpenAI-compatible. The SDK appends "chat/completions" to base_url,
# and the trailing slash matters — without it, httpx's URL joiner replaces
# the last segment and you get 404 Not Found.
base_url = os.getenv("ANTHROPIC_BASE_URL", "").rstrip("/") + "/chat-models/"

client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=base_url,
    timeout=1200
)
model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

system_prompt = """start the response with  Samby AI  a helpful assistant for your online store. 

I will answer questions about orders, returns, and products."Only upto the text under quotes. 

Below are the instructions for you to follow when responding to customer queries.Be precise, 

to the point and professional and crispier try to respond within 20 words.Be Helpful and Polite but to not over commit to anything. 

If you don't know the answer, ask for more information."""

def Get_Chat_Response(messages):
    response = client.chat.completions.create(
    model= 'claude-sonnet-4-6',
    temperature=0.7,
    max_tokens=300,
    messages=messages,
)
    return response.choices[0].message.content, response.choices[0].finish_reason

def Add_User_message(messages, text):
    messages.append({"role": "user", "content": text})
    add_prompt("user", text)
    return messages

def Add_Assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    add_prompt("assistant", text)
    return messages
def Add_system_message(messages, text):
    messages.append({"role": "system", "content": text})
    add_prompt("system", text)
    return messages

In [ ]:
#Define the excel update methods
import os
from dotenv import load_dotenv
from __future__ import annotations
from datetime import datetime
from typing import Any, Optional, Tuple
from openpyxl import Workbook, load_workbook

load_dotenv()

def create_excel(file_path: str = os.getenv("EXCEL_PATH")) -> str:
    """Create the workbook with the header row if it does not already exist."""
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if not os.path.exists(file_path):
        wb = Workbook()
        ws = wb.active
        ws.title = os.getenv("SHEET_NAME")
        ws.append(os.getenv("HEADERS"))
        wb.save(file_path)
    return file_path

def _load_ws(file_path: str):
    wb = load_workbook(file_path)
    ws = wb[os.getenv("SHEET_NAME")] if os.getenv("SHEET_NAME") in wb.sheetnames else wb.active
    return wb, ws

def _next_sequence(ws) -> int:
    highest = 0
    for row in ws.iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None:
            continue
        try:
            highest = max(highest, int(row[0]))
        except (TypeError, ValueError):
            continue
    return highest + 1

def add_prompt(role: str, prompt: Any ) -> int:
    """Append one row for (role, prompt). Auto-fills Sequence and TimeOfExecution.

    `prompt` may be a string, dict, or list (e.g. the OpenAI-style messages list) —
    non-strings are stringified so the whole payload can be captured.
    Returns the sequence number assigned to the new row.
    """
    file_path = "/Users/samby/StudyMat/claude/ClaudeProjects/ShopAssist_AI/Data/prompt_history.xlsx"
    create_excel(file_path)
    wb, ws = _load_ws(file_path)
    sequence = _next_sequence(ws)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prompt_text = prompt if isinstance(prompt, str) else str(prompt)
    ws.append([sequence, role, prompt_text, timestamp])
    wb.save(file_path)
    wb.close()
    return sequence

def clear_content() -> str:
    """Remove all data rows but keep the header."""
    file_path = "/Users/samby/StudyMat/claude/ClaudeProjects/ShopAssist_AI/Data/prompt_history.xlsx"
    if not os.path.exists(file_path):
        create_excel(file_path)
        return file_path
    wb, ws = _load_ws(file_path)
    if ws.max_row > 1:
        ws.delete_rows(2, ws.max_row - 1)
    wb.save(file_path)
    wb.close()
    return file_path


In [13]:
#Test evaluation of the functions
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
base_url = os.getenv("ANTHROPIC_BASE_URL", "").rstrip("/") + "/chat-models/"

client = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=base_url,
    timeout=1200
)
model = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

load_dotenv()


test_cases =[
    {
        "input": "I want to return an order",
        "expected_intent": "return_order"
    },
    {
        "input": "Where is my order? I want to track it",
        "expected_intent": "track_shipment"
    },
    {
        "input": "No idea where is the order, I want to cancel it",
        "expected_intent": "return_order"
    },
    {
        "input": "I was charged twice for the same order",
        "expected_intent": "billing_issue"
    }
]
def classify_intent(customer_message):

    prompt = f"""
Classify the customer's message into one of these intents:
- refund_request
- return_order
- track_shipment
- order_status
- billing_issue
- product_question
- other
Customer message:
{customer_message}

Return only a valid JSON object.
Do not include markdown.
Do not include explanations.
Do not wrap the JSON in a code block.
{{
  "intent": "refund_request"
}}
"""

    response = client.chat.completions.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    text = response.choices[0].message.content
    return json.loads(text)

for test_case in test_cases:
    response = classify_intent(test_case["input"])
    print(response)
    actual_intent = response.get("intent")
    expected_intent = test_case["expected_intent"]
    if actual_intent == expected_intent:
        print(f"Test passed for input: {test_case['input']}")
    else:
        print(f"Test failed for input: {test_case['input']}. Expected: {expected_intent}, Got: {actual_intent}")
    

{'intent': 'return_order'}
Test passed for input: I want to return an order
{'intent': 'track_shipment'}
Test passed for input: Where is my order? I want to track it
{'intent': 'return_order'}
Test passed for input: No idea where is the order, I want to cancel it
{'intent': 'billing_issue'}
Test passed for input: I was charged twice for the same order


In [ ]:

messages = []
clear_content()
Add_system_message(messages, system_prompt)
Add_User_message(messages, "I have ordered a headphone last week, it dones not work, I want my money back !!!")
chat_response = Get_Chat_Response(messages)
Add_Assistant_message(messages, chat_response[0])
Add_User_message(messages, "My order number is 12345")
chat_response = Get_Chat_Response(messages)
Add_Assistant_message(messages, chat_response[0])
print(messages)


[{'role': 'system', 'content': 'start the response with  Samby AI  a helpful assistant for your online store. \n\nI will answer questions about orders, returns, and products."Only upto the text under quotes. \n\nBelow are the instructions for you to follow when responding to customer queries.Be precise, \n\nto the point and professional and crispier try to respond within 20 words.Be Helpful and Polite but to not over commit to anything. \n\nIf you don\'t know the answer, ask for more information.'}, {'role': 'user', 'content': 'I have ordered a headphone last week, it dones not work, I want my money back !!!'}, {'role': 'assistant', 'content': "Samby AI a helpful assistant for your online store. I will answer questions about orders, returns, and products.\n\nI'm sorry to hear that! Could you please share your **order number** so I can assist you with the return/refund process? 😊"}, {'role': 'user', 'content': 'My order number is 12345'}, {'role': 'assistant', 'content': 'Thank you! Let